# Load packages

In [1]:
import os
import ee
import geemap
import json
import requests
import folium

from IPython.display import display, HTML, clear_output

# Initialise GEE

You will need to authenticate your Google account the first time you run this. 

In [2]:
# Initialize Earth Engine
try:
    ee.Initialize()
    print("Earth Engine already initialized")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("Earth Engine initialized")

Earth Engine already initialized


## Plot the basemap

The first time you run the code below, follow the link to authenticate your Google account. After following the instructions online, you will receive an authorization code, which you can paste back into the input box in your notebook. If you're using VSCode, this will appear at the top of your window where the search bar is.

In [3]:
Map = geemap.Map(lite_mode=True, zoom=2)
Map.add_basemap("SATELLITE")
# Map

# Clean up existing HTML files
html_files = ['satellite_map.html', 'satellite_map2.html', 'satellite_map_S2.html']
for file in html_files:
    if os.path.exists(file):
        os.remove(file)

# Clear previous outputs before displaying new map
clear_output(wait=True)
Map.to_html(filename='satellite_map.html')
HTML('satellite_map.html')

## Import AOI

In [4]:
# Option 1: Read JSON from file
def load_aoi_from_file(json_file_path):
    with open(json_file_path, 'r') as f:
        geojson = json.load(f)
    
    # Create an ee.Geometry from the GeoJSON
    return ee.Geometry(geojson)

In [6]:
# Alternative: Define AOI from coordinates
# aoi = ee.Geometry.Rectangle([-122.5, 37.5, -122.0, 38.0])  # San Francisco area

# Load AOI from file
aoi_name = 'mimal_test'
aoi = load_aoi_from_file(fr'AOIs/{aoi_name}.geojson')

# New basemap for AOI
Map2 = geemap.Map(lite_mode=True, zoom=2)
Map2.add_basemap("SATELLITE")
Map2

# Show the AOI on map
Map2.addLayer(aoi, {}, 'Area of Interest')
# Center the map on the AOI and zoom to it
Map2.centerObject(aoi, zoom=10)  # Adjust zoom level (1-20) as needed
# Map  # Display the map with the AOI

# Clear previous outputs before displaying new map
clear_output(wait=True)
Map2.to_html(filename='satellite_map2.html')
HTML('satellite_map2.html')

# Get Sentinel-2 Collection with Cloud Masking

This function creates a cloud mask for Sentinel-2 imagery.

There are other approaches to filter clouds, such as the approach listed here: [https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless](https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless).

In [26]:
def maskS2clouds_CSPlus(image):
    """
    Mask Sentinel-2 using Cloud Score+.
    Assumes the CS+ bands ('cs', 'cs_cdf') have already been linked
    to the S2 collection via linkCollection().
    """
    # Use the cs_cdf band (cumulative distribution function variant)
    # is generally more robust than 'cs' for time-series work.
    # Threshold range: 0 (not clear) to 1 (clear).
    #   0.60 = permissive (more pixels kept, some haze/thin cloud)
    #   0.65 = balanced (Google's commonly recommended default)
    #   0.80+ = strict (clean composites, fewer observations)
    QA_BAND = 'cs_cdf'
    CLEAR_THRESHOLD = 0.65

    mask = image.select(QA_BAND).gte(CLEAR_THRESHOLD)
    return (image.divide(10000)
                 .updateMask(mask)
                 .copyProperties(image, ['system:time_start']))

# Get Monthly Composites of Sentinel-2

## Define a function to get monthly composites

We want to create monthly composites of Sentinel-2 imagery. This function will filter the Sentinel-2 image collection by date and area of interest (AOI), apply the cloud mask, and then compute the median of each 'cloud-free' pixel for the month.

## Instead, get a temporal range longer than a month and take the median

In [48]:
def _build_s2_composites(start_date, end_date, aoi, composite_type='monthly', band_mode='rgb'):
    """Build Sentinel-2 composites as monthly series or a single period median."""
    composite_type = composite_type.lower()
    bands, band_suffix = get_s2_band_selection(band_mode)

    s2_base = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(aoi)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 90)))

    csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    s2_linked = s2_base.linkCollection(csPlus, ['cs', 'cs_cdf'])

    raw_count = s2_base.size().getInfo()
    print(f"  Found {raw_count} S2 images for {start_date} to {end_date}")

    if composite_type == 'period':
        masked = s2_linked.map(maskS2clouds_CSPlus)
        composite = masked.median().clip(aoi).select(bands)
        period_id = f'{start_date}_to_{end_date}'
        period_time = ee.Date(start_date).format('YYYY-MM').cat('_to_').cat(ee.Date(end_date).format('YYYY-MM'))
        return ee.ImageCollection([composite.set({
            'system:index': period_id,
            'system:time': period_time,
            'band_mode': band_mode,
            'band_suffix': band_suffix,
            'composite_type': composite_type,
        })])

    if composite_type != 'monthly':
        raise ValueError("composite_type must be 'monthly' or 'period'")

    start = ee.Date(start_date)
    end = ee.Date(end_date)
    months = end.difference(start, 'month').round().int()

    def get_monthly_image(month_index):
        current_month_start = start.advance(month_index, 'month')
        current_month_end = current_month_start.advance(1, 'month')
        date_format = current_month_start.format('YYYY-MM')
        # month_time = current_month_start.format('YYYY-MM-dd').cat('_to_').cat(current_month_end.format('YYYY-MM-dd'))

        monthly = (s2_linked
                     .filterDate(current_month_start, current_month_end)
                     .map(maskS2clouds_CSPlus))

        composite = monthly.median().clip(aoi).select(bands)
        return composite.set({
            'system:index': date_format,
            'system:time': date_format,
            'band_mode': band_mode,
            'band_suffix': band_suffix,
            'composite_type': composite_type,
        })

    month_indices = ee.List.sequence(0, months.subtract(1))
    return ee.ImageCollection.fromImages(month_indices.map(get_monthly_image))


# # Backwards-compatible wrappers.
# def get_monthly_composites(start_date, end_date, aoi, band_mode='rgb'):
#     return _build_s2_composites(start_date, end_date, aoi, composite_type='monthly', band_mode=band_mode)


# def get_period_composite(start_date, end_date, aoi, band_mode='rgb'):
#     return _build_s2_composites(start_date, end_date, aoi, composite_type='period', band_mode=band_mode)

## Export to Google Drive

# Download images for a specified date range

We will get an image as an example.

In [49]:
# Define parameters
start_date = '2024-06-01'
end_date = '2024-12-31' 
output_folder = 'image/sentinel2_monthly_images'

### Run the function

The function will print how many suitable images were found for the specified date range and AOI. If no images are found, you may need to adjust your date range or AOI.

In [50]:
import os
import zipfile
from pathlib import Path

S2_BAND_SETS = {
    'rgb': ['B4', 'B3', 'B2'],
    'full': ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12'],
}


def get_s2_band_selection(band_mode='rgb'):
    band_mode = band_mode.lower()
    if band_mode not in S2_BAND_SETS:
        raise ValueError(f"band_mode must be one of {sorted(S2_BAND_SETS)}")
    return S2_BAND_SETS[band_mode], band_mode


# Export monthly or period composites either locally or to Google Drive.
def export_composites(composites, aoi, aoi_name, destination='local', output_folder=None, drive_folder='Earth_Engine_Exports', scale=10):
    destination = destination.lower()
    composite_list = composites.toList(composites.size())
    num_images = composites.size().getInfo()

    if destination == 'local':
        if output_folder is None:
            raise ValueError('output_folder is required when destination="local"')
        output_folder = Path(output_folder)
        output_folder.mkdir(parents=True, exist_ok=True)

    for i in range(num_images):
        image = ee.Image(composite_list.get(i))
        time_label = image.get('system:time').getInfo()
        band_suffix = image.get('band_suffix').getInfo()
        save_name = f"{aoi_name}_{time_label}_{band_suffix}"

        if destination == 'local':
            url = image.getDownloadURL({
                'scale': scale,
                'region': image.geometry().bounds().getInfo()['coordinates'],
                'format': 'GEO_TIFF'
            })

            zip_path = output_folder / f"{save_name}.zip"
            tif_path = output_folder / f"{save_name}.tif"

            response = requests.get(url)
            response.raise_for_status()
            zip_path.write_bytes(response.content)

            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                tif_members = [name for name in zip_ref.namelist() if name.lower().endswith('.tif')]
                member_name = tif_members[0] if tif_members else zip_ref.namelist()[0]
                extracted_path = Path(zip_ref.extract(member_name, output_folder))

            extracted_path.replace(tif_path)
            zip_path.unlink(missing_ok=True)
            print(f"Saved {tif_path}")
            continue

        if destination == 'drive':
            task = ee.batch.Export.image.toDrive(
                image=image,
                description=save_name,
                folder=drive_folder,
                fileNamePrefix=save_name,
                region=aoi,
                scale=scale,
                maxPixels=1e13
            )
            task.start()
            print(f"Started Drive export: {save_name}")
            continue

        raise ValueError("destination must be 'local' or 'drive'")

In [51]:
print(f'AOI name:   {aoi_name}')

band_mode = 'full'  # change to 'full' to export the full Sentinel-2 optical stack
composite_type = 'period'  # change to 'period' for a single median composite over the full date range
composites = _build_s2_composites(start_date, end_date, aoi, composite_type=composite_type, band_mode=band_mode)

AOI name:   mimal_test
  Found 84 S2 images for 2024-06-01 to 2024-12-31


# Export 

In [52]:
export_composites(composites, aoi, aoi_name, destination='drive', drive_folder='sentinel2_images', scale=10)

Started Drive export: mimal_test_2024-06_to_2024-12_full
